# Generation of Read Pileups — Guided Exercises

**AMLG | Read simulation, sequencing depth, coverage, FASTA files, and introductory variant calling**

This notebook is designed for a live teaching session **and independent self-study**. You will start with a short DNA reference, simulate sequencing reads, stack them into a *pileup*, and measure how well the reference is covered. You will then apply the same workflow to a short region from a real *E. coli* genome. An **optional extension** demonstrates sequencing errors, simple read mapping, and a majority-vote SNP call.

> **Scope:** The original exercise covered read sampling, visual alignment using known positions, pileup depth, mean coverage, FASTA parsing, and *E. coli*.  This guided version keeps the original learning goals and adds a small, clearly marked extension rather than implying that full alignment algorithms are included.

## How to use this notebook

1. Run cells from top to bottom with **Shift + Enter**.
2. Read each **Task**, **Hint**, and **Expected outcome** before editing code.
3. Complete each `# TODO` yourself. The worked answer is provided as **commented-out code** under `# --- Solution (uncomment to check) ---`.
4. To reveal a solution, uncomment its lines and remove any placeholder `pass` or `raise NotImplementedError` first.
5. Run the cell again and compare its output with the **Expected outcome**. Different random reads are normal unless a seed is used.

**Learning route:** DNA reference → sample reads → display reads at known positions → pileup/depth → mean coverage → visualization → read FASTA → *E. coli* example → **optional:** introduce SNP/errors → map reads → majority-vote SNP calling.

**Prerequisites:** Basic Python (`for` loops, strings, lists, dictionaries) and NumPy/Matplotlib. No specialist biology knowledge is assumed.

## 0. Setup and key ideas

DNA uses four bases: `A`, `C`, `G`, and `T`. A **reference** is a longer known DNA sequence. A **read** is a shorter DNA sequence produced by sequencing. Multiple reads can cover the same reference position.

In this exercise, each read is a dictionary such as `{'pos': 4, 'seq': 'TTTCATT'}`: `pos` is its **0-based** start position in the reference, and `seq` contains its DNA bases. For now we know the true start because **we generated the reads ourselves**. Recovering that position from an unknown read is called *read mapping* and is introduced only in the optional extension.

**Pileup / depth:** At position `i`, depth is the number of reads spanning `i`. **Mean coverage** in this exercise means the *average depth over all reference positions*, including positions covered by zero reads. (In some contexts, “breadth of coverage” instead means the fraction of positions with at least one read.)

A practical rule for a read starting at `pos` with length `L`: it covers Python indices `pos, pos+1, ..., pos+L-1`, which is the half-open slice `[pos : pos+L]`.

In [ ]:
import random
from pathlib import Path
from typing import Dict, List

import numpy as np
import matplotlib.pyplot as plt

RNG_SEED = 42
random.seed(RNG_SEED)
print("Setup complete — random seed:", RNG_SEED)

## 1. Sample sequencing reads

Imagine cutting short pieces out of the reference DNA at randomly chosen places. This simulates **error-free sequencing**: each sampled read matches the reference exactly.

### ❓ Exercise 1 — Implement `sample_reads()`

**Task:** Return `n_reads` dictionaries, each containing a random read length between `min_read_len` and `max_read_len` (inclusive), a valid 0-based start `pos`, and the matching DNA substring `seq`.

**Hint:** `random.randint(a, b)` includes both endpoints. The last valid start for a read of length `L` is `len(reference_sequence) - L`. Retrieve the substring with `reference_sequence[start:start+L]`. Check that length bounds are valid.

**Expected outcome:** A list of exactly `n_reads` dictionaries. Every read is within the reference and perfectly matches the corresponding substring.

**What this teaches:** How a sequencing read is represented, and how reference coordinates work.

In [ ]:
def sample_reads(reference_sequence: str, n_reads: int,
                 min_read_len: int, max_read_len: int) -> List[Dict]:
    """Sample error-free reads and record their true reference start positions."""
    if not (0 < min_read_len <= max_read_len <= len(reference_sequence)):
        raise ValueError("Read lengths must fit inside the reference")
    if n_reads < 0:
        raise ValueError("n_reads must be nonnegative")
    reads = []
    # TODO: Loop n_reads times. Draw a length and valid start; append {'pos': ..., 'seq': ...}.
    raise NotImplementedError("Complete Exercise 1, or uncomment its solution")

    # --- Solution (uncomment to check) ---
    # for _ in range(n_reads):
    #     read_len = random.randint(min_read_len, max_read_len)
    #     start = random.randint(0, len(reference_sequence) - read_len)
    #     reads.append({'pos': start, 'seq': reference_sequence[start:start + read_len]})
    # return reads

### ❓ Exercise 2 — Generate a small sequencing experiment

**Task:** Use the provided reference and call `sample_reads()` with `n_reads=20`, `min_read_len=15`, and `max_read_len=40`. Print the first three dictionaries.

**Hint:** Assign the result to `reads`. Index with `reads[:3]`.

**Expected outcome:** Twenty reads; each length is between 15 and 40 bases. The precise sequences depend on the random seed.

**What this teaches:** How simulation parameters control the input to downstream analysis.

In [ ]:
reference_sequence = (
    "AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATTAAAAAAAGAGTGTCTGATAGCAGC"
)
# TODO: Create the variable `reads` and print the first three reads.
raise NotImplementedError("Complete Exercise 2, or uncomment its solution")

# --- Solution (uncomment to check) ---
# reads = sample_reads(reference_sequence, n_reads=20, min_read_len=15, max_read_len=40)
# print('Number of reads:', len(reads))
# for read in reads[:3]:
#     print(read)

### ❓ Exercise 3 — Show the reads lined up with the reference

**Task:** Print the reference, then each read on a separate line indented by its `pos` value (spaces before the DNA sequence).

**Hint:** Multiplying a string repeats it: `' ' * 4` creates four spaces. `enumerate(reads)` gives the read number.

**Expected outcome:** Reads appear horizontally below the DNA region from which they originated. Example shape:

```text
Reference: AGCTTTTCATTCTGACTGCA...
Read    0:     TTTCATTCTGACTGCA...
```

**What this teaches:** A visual *alignment* when true mapping positions are already known. **This does not map unknown reads.**

In [ ]:
# TODO: Print the reference and every read with the appropriate leading spaces.
raise NotImplementedError("Complete Exercise 3, or uncomment its solution")

# --- Solution (uncomment to check) ---
# print('Reference:', reference_sequence)
# for i, read in enumerate(reads):
#     print(f"Read {i:4d}: "+ ' ' * read['pos'] + read['seq'])

## 2. From overlapping reads to depth and coverage

A **pileup** groups the read bases that overlap a particular reference position. The **pileup size** (also called **sequencing depth**) counts how many reads span that position, independent of which base they contain.

For example, three reads overlapping position 5 produce depth `3` there. A position with no reads has depth `0`.

### ❓ Exercise 4 — Compute pileup depth at every position

**Task:** Finish `compute_pileup_sizes()`. Return one depth value for each reference position.

**Hint:** Start with `np.zeros(len(reference_sequence), dtype=int)`. For each read, increase the slice `depth[pos:pos + len(seq)]` by one.

**Expected outcome:** A 1D array with length equal to the reference length. Every value is a nonnegative integer. `sum(depth)` equals the total number of sequenced bases across all reads (if all reads stay within bounds).

**What this teaches:** The connection between read overlaps and per-base depth.

In [ ]:
def compute_pileup_sizes(reference_sequence: str, reads: List[Dict]) -> np.ndarray:
    """Return sequencing depth for each 0-based position in the reference."""
    pileup_sizes = np.zeros(len(reference_sequence), dtype=int)
    # TODO: Loop over reads and increment the covered slice.
    raise NotImplementedError("Complete Exercise 4, or uncomment its solution")

    # --- Solution (uncomment to check) ---
    # for read in reads:
    #     start = read['pos']
    #     end = start + len(read['seq'])
    #     pileup_sizes[start:end] += 1
    # return pileup_sizes

# TODO: Compute pileup_sizes from reference_sequence and reads, then print it.
# --- Solution (uncomment to check) ---
# pileup_sizes = compute_pileup_sizes(reference_sequence, reads)
# print(pileup_sizes)
# assert len(pileup_sizes) == len(reference_sequence)
# assert pileup_sizes.sum() == sum(len(r['seq']) for r in reads)

### ❓ Exercise 5 — Calculate average depth (mean coverage)

**Task:** Write `coverage(pileup_sizes)` to return the **mean** of the depth values.

**Hint:** Use `np.mean(pileup_sizes)`. This average includes zero-depth positions. A region with depth values `[2, 0, 1]` has mean depth `1.0`.

**Expected outcome:** A nonnegative decimal number such as `5.32`. There is no universal fixed result; it depends on the sampled reads.

**What this teaches:** The difference between **depth at one position** and **average depth across a region**.

In [ ]:
def coverage(pileup_sizes: np.ndarray) -> float:
    """Return mean sequencing depth across all reference positions."""
    # TODO: Return the average depth as a float.
    raise NotImplementedError("Complete Exercise 5, or uncomment its solution")

    # --- Solution (uncomment to check) ---
    # return float(np.mean(pileup_sizes))

# TODO: Print coverage with two decimal places.
# --- Solution (uncomment to check) ---
# print(f"Mean coverage: {coverage(pileup_sizes):.2f}x")

### ❓ Exercise 6 — Visualize sequencing depth

**Task:** Make a bar plot of sequencing depth at each reference position, and draw a horizontal line showing the mean depth.

**Hint:** `plt.bar(np.arange(len(pileup_sizes)), pileup_sizes)` and `plt.axhline(coverage(pileup_sizes), ...)`.

**Expected outcome:** A plot whose x-axis is reference position, y-axis is depth, and horizontal line indicates mean coverage.

**What this teaches:** Mean coverage can look acceptable even when some positions have very little or no data.

In [ ]:
def plot_pileups(reference_sequence: str, pileup_sizes: np.ndarray) -> None:
    """Plot per-position depth and mean depth."""
    plt.figure(figsize=(11, 4))
    # TODO: Add bars, a horizontal mean-depth line, labels, and a legend.
    raise NotImplementedError("Complete Exercise 6, or uncomment its solution")

    # --- Solution (uncomment to check) ---
    # plt.bar(np.arange(len(pileup_sizes)), pileup_sizes, width=1.0, label='Depth')
    # plt.axhline(coverage(pileup_sizes), linestyle='--', color='tab:orange',
    #             label=f'Mean: {coverage(pileup_sizes):.2f}x')
    # plt.xlabel('Reference position (0-based)')
    # plt.ylabel('Sequencing depth (reads)')
    # plt.title('Read pileup depth across reference')
    # plt.legend()
    # plt.tight_layout()
    # plt.show()

# TODO: Call plot_pileups(reference_sequence, pileup_sizes).
# --- Solution (uncomment to check) ---
# plot_pileups(reference_sequence, pileup_sizes)

### 🔎 Quick self-check: What do these numbers mean?

Before continuing, answer these questions in your own words:

1. If you double the number of randomly sampled reads, what do you expect to happen to **average** depth?
2. Does average depth of `10x` guarantee that **every** reference position has ten reads?
3. How is **mean depth** different from **breadth of coverage** (`fraction of positions with depth > 0`)?

**Check yourself:** (1) approximately doubles; (2) no, depth varies by position; (3) they summarize different things—the average number of reads versus the fraction of positions covered at least once.

## 3. Working with a real *E. coli* reference FASTA

A **FASTA** file stores one or more sequences. A record begins with a header line starting with `>`, followed by one or more lines of sequence. The header identifies the record; the sequence lines should be concatenated into one continuous DNA string.

The original class exercise uses the *Escherichia coli* DH10B chromosome file `e-coli-dh10b.fasta`. We will only simulate reads from its **first 1,000 bases**, so the activity remains small and fast.

**Data setup:** If the file is already in `data/`, no download is needed. The next cell attempts the course-provided download when missing; internet access may be required. The exercises in Sections 1–2 work offline.

In [ ]:
DATA_PATH = Path('data/e-coli-dh10b.fasta')
DATA_URL = (
    'https://seafile.cloud.uni-hannover.de/d/5d6029c6eaaf410c8b01/'
    'files/?p=%2Fread_pileups%2Fe-coli-dh10b.fasta&dl=1'
)

if DATA_PATH.exists():
    print('FASTA available:', DATA_PATH)
else:
    print('FASTA not found. Attempting course download...')
    try:
        from urllib.request import urlretrieve
        DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(DATA_URL, DATA_PATH)
        print('Downloaded:', DATA_PATH)
    except Exception as exc:
        print('Download failed:', exc)
        print('For Section 3, manually place e-coli-dh10b.fasta in data/.')

### ❓ Exercise 7 — Write a FASTA parser

**Task:** Complete `read_fasta_file()`. Return a dictionary mapping **header descriptions (without the `>`)** to full sequence strings. Support multiple FASTA records and sequences split across several lines.

**Hint:** Strip whitespace. When a line starts with `>`, create a new dictionary entry; otherwise append the sequence line to the current entry. Skip empty lines. A sequence line before a header is invalid.

**Expected outcome:** A dictionary of descriptions and DNA strings. If the course FASTA is available, print the record names and genome length.

**What this teaches:** How raw reference sequences are stored and imported for sequence analysis.

In [ ]:
def read_fasta_file(file_path: str) -> Dict[str, str]:
    """Parse a FASTA file into {description: concatenated_sequence}."""
    sequences: Dict[str, str] = {}
    current_description = None
    with open(file_path, 'r', encoding='utf-8') as fasta:
        for raw_line in fasta:
            line = raw_line.strip()
            if not line:
                continue
            # TODO: Detect header lines and append ordinary lines to the current sequence.
            raise NotImplementedError("Complete Exercise 7, or uncomment its solution")

            # --- Solution (uncomment to check) ---
            # if line.startswith('>'):
            #     current_description = line[1:]
            #     sequences[current_description] = ''
            # else:
            #     if current_description is None:
            #         raise ValueError('Sequence found before a FASTA header')
            #     sequences[current_description] += line
    return sequences

# TODO: Once FASTA exists, load it, inspect its headers, and select its chromosome sequence.
# --- Solution (uncomment to check) ---
# ecoli_genome = read_fasta_file(str(DATA_PATH))
# print('FASTA records:', list(ecoli_genome))
# ecoli_genome_sequence = next(iter(ecoli_genome.values()))
# print(f'E. coli reference length: {len(ecoli_genome_sequence):,} bp')

### ❓ Exercise 8 — Simulate and visualize *E. coli* pileups

**Task:** Take the first `1,000` bases of the imported *E. coli* genome, simulate `1,000` reads of lengths `100–250`, compute per-base depth, and plot it with its average-depth line.

**Hint:** Reuse your earlier functions and slice the genome with `ecoli_genome_sequence[:1000]`. Do not simulate from all ~4.6 million bases here.

**Expected outcome:** A 1,000-position depth plot; average depth is broadly around ~175x, with a lower depth near the ends because sampled reads must fit entirely in the window. The exact value varies with randomness.

**What this teaches:** The same small Python functions can analyze a longer real reference sequence. Observe the **edge effect** caused by sampling only within a fixed window.

In [ ]:
# TODO: Produce the simulated E. coli experiment with your functions.
raise NotImplementedError("Complete Exercise 8, or uncomment its solution")

# --- Solution (uncomment to check) ---
# ecoli_region = ecoli_genome_sequence[:1000]
# ecoli_reads = sample_reads(ecoli_region, n_reads=1000, min_read_len=100, max_read_len=250)
# ecoli_depth = compute_pileup_sizes(ecoli_region, ecoli_reads)
# print(f'Mean E. coli depth: {coverage(ecoli_depth):.2f}x')
# plot_pileups(ecoli_region, ecoli_depth)

## 4. Optional extension — Read mapping and a basic SNP call

**New material (not in the original exercise).** Up to this point, each simulated read stored its real position. In sequencing experiments the position is not known in advance: a read **must be mapped** to the reference. Here we use a deliberately simple method for short, equal-length DNA comparisons.

We also distinguish two concepts:

- A **variant / SNP** is a true difference between the sequenced sample and the reference at one base.
- A **sequencing error** is a wrong base observed in an individual read.

This extension uses a tiny synthetic example with a fixed read length and substitutions only. It does **not** handle insertions, deletions, reverse-complement reads, alignment scoring, ambiguous repeats, or calibrated variant quality. Real sequencing tools use more sophisticated aligners and variant callers.

### ❓ Exercise 9 — Introduce a true SNP and noisy reads

**Task:** Create a sample genome by changing one chosen reference base to another. Generate reads from that sample genome with known positions, then introduce occasional random substitution errors in their sequences. **Keep their true positions** for this step.

**Hint:** Choose a site well inside a short reference. To replace one base in a string use `seq[:i] + new_base + seq[i+1:]`. For errors, each base can be replaced with probability `error_rate`; choose a different base from `'ACGT'`.

**Expected outcome:** The sample DNA differs from the reference at exactly one position, whereas some reads also contain additional random mistakes.

**What this teaches:** Why one mismatching read is not enough to establish a real SNP.

In [ ]:
tiny_reference = 'ACGTTGCAAGTCCTAGGCTAATCGGTACCTAGTCAAGTCCGATCGATGCTACGTAGCTAG'
variant_pos = 30  # 0-based
original_base = tiny_reference[variant_pos]
variant_base = next(base for base in 'ACGT' if base != original_base)

# TODO: Create `sample_genome` with one base change, then simulate 120 reads (length 18).
# TODO: Create `noisy_reads` by mutating read bases with 2% probability.
raise NotImplementedError("Complete Exercise 9, or uncomment its solution")

# --- Solution (uncomment to check) ---
# sample_genome = (tiny_reference[:variant_pos] + variant_base
#                  + tiny_reference[variant_pos+1:])
# clean_reads = sample_reads(sample_genome, 120, 18, 18)
# noisy_reads = []
# for read in clean_reads:
#     noisy_seq = []
#     for base in read['seq']:
#         if random.random() < 0.02:
#             base = random.choice([b for b in 'ACGT' if b != base])
#         noisy_seq.append(base)
#     noisy_reads.append({'pos': read['pos'], 'seq': ''.join(noisy_seq)})
# print('Variant:', variant_pos, original_base, '->', variant_base)
# print('Example noisy read:', noisy_reads[0])

### ❓ Exercise 10 — Map by minimum mismatches (simplified)

**Task:** Implement `map_read_by_mismatches(read_seq, reference)`. Compare the read against *every* equal-length window in the reference and return the start position with the fewest mismatching characters.

**Hint:** Count mismatches with `sum(a != b for a, b in zip(read_seq, window))`. Track the smallest mismatch count and its start. A variant or sequencing error can cause mismatches even at the correct mapping location.

**Expected outcome:** A tuple `(start, mismatch_count)`. For many reads from this toy example, the predicted start equals the known `pos`; ties/repetitive sequences can still make this approach ambiguous.

**What this teaches:** A basic read-mapping idea. This is **not** global/local dynamic-programming alignment.

In [ ]:
def map_read_by_mismatches(read_seq: str, reference: str):
    """Return (best_start, fewest_mismatches) for equal-length ungapped placement."""
    if len(read_seq) > len(reference):
        raise ValueError('Read longer than reference')
    # TODO: Scan every possible start and find the minimum mismatch count.
    raise NotImplementedError("Complete Exercise 10, or uncomment its solution")

    # --- Solution (uncomment to check) ---
    # best_start, best_mismatches = None, len(read_seq) + 1
    # for start in range(len(reference) - len(read_seq) + 1):
    #     window = reference[start:start + len(read_seq)]
    #     mismatches = sum(a != b for a, b in zip(read_seq, window))
    #     if mismatches < best_mismatches:
    #         best_start, best_mismatches = start, mismatches
    # return best_start, best_mismatches

# TODO: Try this function on the first 5 noisy reads; compare predicted to true starts.
# --- Solution (uncomment to check) ---
# for r in noisy_reads[:5]:
#     predicted, mismatches = map_read_by_mismatches(r['seq'], tiny_reference)
#     print(f"true={r['pos']:2d}, mapped={predicted:2d}, mismatches={mismatches}")

### ❓ Exercise 11 — Pile up bases and call a SNP

**Task:** Map the noisy reads to the reference using Exercise 10. For each reference position, count observed `A/C/G/T` from reads mapped across it. At positions with at least **5 supporting reads**, call a candidate SNP if the most frequent base differs from the reference and its frequency is at least **0.70**.

**Hint:** Use one `dict` (or `collections.Counter`) per position. For each mapped read at `start`, the base at read offset `j` contributes to reference position `start + j`. Compare the majority base and total depth.

**Expected outcome:** A candidate near the planted SNP (at position `30`) and few or no false calls, depending on noise and mapping. This toy majority-vote rule is not a validated real-world SNP caller.

**What this teaches:** How a pileup can store **base identities**, not just depth, and support a simple variant decision.

In [ ]:
from collections import Counter

def base_pileup(reference: str, read_sequences: List[str]) -> List[Counter]:
    """Count observed bases at reference positions after simple read mapping."""
    counts = [Counter() for _ in reference]
    # TODO: Map each read; accumulate counts at covered positions.
    raise NotImplementedError("Complete Exercise 11, or uncomment its solution")

    # --- Solution (uncomment to check) ---
    # for seq in read_sequences:
    #     start, _ = map_read_by_mismatches(seq, reference)
    #     for offset, base in enumerate(seq):
    #         counts[start + offset][base] += 1
    # return counts

def majority_snp_calls(reference: str, counts: List[Counter],
                       min_depth: int = 5, min_fraction: float = 0.70):
    """Return (position, ref_base, candidate_base, depth, fraction) calls."""
    calls = []
    # TODO: Check depth and most-common base at each position; append qualifying SNPs.
    raise NotImplementedError("Complete Exercise 11, or uncomment its solution")

    # --- Solution (uncomment to check) ---
    # for pos, counter in enumerate(counts):
    #     depth = sum(counter.values())
    #     if depth < min_depth:
    #         continue
    #     base, n = counter.most_common(1)[0]
    #     fraction = n / depth
    #     if base != reference[pos] and fraction >= min_fraction:
    #         calls.append((pos, reference[pos], base, depth, round(fraction, 3)))
    # return calls

# TODO: Build the base pileup and print called SNPs.
# --- Solution (uncomment to check) ---
# counts = base_pileup(tiny_reference, [r['seq'] for r in noisy_reads])
# calls = majority_snp_calls(tiny_reference, counts)
# print('Candidate SNPs:', calls)
# print('Expected planted SNP site:', variant_pos)

## 5. Wrap-up and questions

You have now worked through the original core lesson—**sample reads → visualize overlap → compute depth → measure mean coverage → plot pileups → parse FASTA → apply to *E. coli***—and optionally explored **substitution errors, basic mapping, and SNP calling**.